In [1]:
import pandas as pd

# Load the raw Excel file
raw = pd.read_excel('../data/Online Retail.xlsx', engine='openpyxl')

# Check the shape — how many rows and columns
print(f"Rows: {raw.shape[0]}, Columns: {raw.shape[1]}")
print(f"\nColumn names: {list(raw.columns)}")
print(f"\nFirst 5 rows:")
raw.head()

Rows: 541909, Columns: 8

Column names: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

First 5 rows:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
# Check for missing values in each column
print("Missing values per column:")
print(raw.isnull().sum())

print(f"\nData types:")
print(raw.dtypes)

print(f"\nBasic statistics:")
raw.describe()

Missing values per column:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Data types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Basic statistics:


,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [3]:
# Start with a working copy — never modify raw
df = raw.copy()

# Drop rows with missing Description
# Reason: only 0.27% of data, negligible revenue impact
# and blank product names would break dashboard visuals
before = len(df)
df = df.dropna(subset=['Description'])
after = len(df)

print(f"Rows before: {before}")
print(f"Rows dropped: {before - after}")
print(f"Rows remaining: {after}")

Rows before: 541909
Rows dropped: 1454
Rows remaining: 540455


In [4]:
# Filter to only cancelled transactions
cancelled = df[df['InvoiceNo'].astype(str).str.startswith('C')]

print(f"Total cancellation rows: {len(cancelled)}")
print(f"\nSample of cancellation rows:")
cancelled.head(10)

Total cancellation rows: 9288

Sample of cancellation rows:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


In [5]:
# Remove cancelled transactions
before = len(df)
df = df[~df.index.isin(cancelled.index)]
print(f"Removed {before - len(df)} cancellations")

# Remove zero or negative quantities
before = len(df)
df = df[df['Quantity'] > 0]
print(f"Removed {before - len(df)} zero/negative quantity rows")

# Remove zero or negative prices
before = len(df)
df = df[df['UnitPrice'] > 0]
print(f"Removed {before - len(df)} zero/negative price rows")

print(f"\nRows remaining: {len(df)}")

Removed 9288 cancellations
Removed 474 zero/negative quantity rows
Removed 589 zero/negative price rows

Rows remaining: 530104


In [6]:
# Standardise Description — strip whitespace, uppercase
df['Description'] = df['Description'].str.strip().str.upper()

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Add TotalAmount column
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

print(f"Data types after cleaning:")
print(df.dtypes)
print(f"\nSample with TotalAmount:")
df[['InvoiceNo', 'Description', 'Quantity', 'UnitPrice', 'TotalAmount']].head()

Data types after cleaning:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
TotalAmount           float64
dtype: object

Sample with TotalAmount:


,InvoiceNo,Description,Quantity,UnitPrice,TotalAmount
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,2.55,15.30
1,536365,WHITE METAL LANTERN,6,3.39,20.34
2,536365,CREAM CUPID HEARTS COAT HANGER,8,2.75,22.00
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,3.39,20.34
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6,3.39,20.34


In [7]:
# Final summary before saving
print(f"Final shape: {df.shape}")
print(f"\nMissing values remaining:")
print(df.isnull().sum())
print(f"\nDate range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")
print(f"\nTotal revenue in dataset: £{df['TotalAmount'].sum():,.2f}")
print(f"Countries represented: {df['Country'].nunique()}")

# Save to CSV
df.to_csv('../data/online_retail_clean.csv', index=False)
print(f"\nClean data saved successfully.")

Final shape: (530104, 9)

Missing values remaining:
InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132220
Country             0
TotalAmount         0
dtype: int64

Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00

Total revenue in dataset: £10,666,684.54
Countries represented: 38

Clean data saved successfully.


In [1]:
from sqlalchemy import create_engine
import urllib

# Connection string for SQL Server using Windows Authentication
# This tells SQLAlchemy:
# - Use the SQL Server driver
# - Connect to localhost\SQLEXPRESS (your local instance)
# - Use the RetailDW database
# - Trust the server certificate (same setting we used in SSMS)
# - Use Windows Authentication (no password needed)

connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=master;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

connection_url = f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(connection_string)}"

engine = create_engine(connection_url)

# Test the connection
with engine.connect() as conn:
    print("Connected to SQL Server successfully")

Connected to SQL Server successfully


C:\Users\rashi\AppData\Local\Temp\ipykernel_11892\1350087459.py:25: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


In [4]:
import pandas as pd
from sqlalchemy import text

# Load the clean CSV — low_memory=False prevents mixed type warning
df = pd.read_csv('../data/online_retail_clean.csv', low_memory=False)
print(f"CSV loaded: {len(df)} rows")

# Create RetailDW database using AUTOCOMMIT
# Reason: CREATE DATABASE cannot run inside a transaction in SQL Server
with engine.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(text("IF NOT EXISTS (SELECT * FROM sys.databases WHERE name = 'RetailDW') CREATE DATABASE RetailDW"))
    print("RetailDW database ready")

# Reconnect pointing to RetailDW specifically
connection_string_dw = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=RetailDW;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
connection_url_dw = f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(connection_string_dw)}"
engine_dw = create_engine(connection_url_dw)

# Load data into SQL Server
print("Loading data into SQL Server — this will take a minute...")
df.to_sql(
    name='retail_sales',
    con=engine_dw,
    if_exists='replace',
    index=False,
    chunksize=1000
)

print(f"Done. {len(df)} rows loaded into RetailDW.retail_sales")

CSV loaded: 530104 rows
RetailDW database ready
Loading data into SQL Server — this will take a minute...


c:\Users\rashi\anaconda3\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Done. 530104 rows loaded into RetailDW.retail_sales
